# Word Boundary Benchmark

This notebook tracks the data-science method for evaluating sentence-level word clipping in Read-Along AI. The app generates one full-sentence VoxCPM helper clip, then slices it into clickable per-word audio. The benchmark asks whether predicted internal word boundaries land inside manually labeled silence gaps.

The important change from the earlier synthetic-only notebook is that this version uses the committed labeled comma-audio dataset under `data/curriculum_audio/comma/`.

## Dataset

The labeled boundary dataset contains four comma-prompted curriculum WAV files and paired Audacity label exports. Each label row has `start`, `end`, and `word`. The manually labeled silence gap between adjacent words is the interval between the previous word end and the next word start.

This is separate from the child-speech MiniCPM training labels: those evaluate ASR and phonetic judging, while this dataset evaluates sentence-audio word slicing.

In [ ]:
from __future__ import annotations

import sys
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import app

COMMA_AUDIO_DIR = REPO_ROOT / "data" / "curriculum_audio" / "comma"


@dataclass(frozen=True)
class WordLabel:
    sentence: str
    wav_path: Path
    label_path: Path
    word: str
    start: float
    end: float


def parse_audacity_labels(sentence: str, wav_path: Path, label_path: Path) -> list[WordLabel]:
    labels: list[WordLabel] = []
    for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
        if not line.strip():
            continue
        parts = line.split("\t")
        if len(parts) != 3:
            raise ValueError(f"{label_path}:{line_number} should have start, end, and word columns")
        start, end, word = parts
        labels.append(
            WordLabel(
                sentence=sentence,
                wav_path=wav_path,
                label_path=label_path,
                word=app.clean_tts_word(word),
                start=float(start),
                end=float(end),
            )
        )
    return labels


def comma_label_cases() -> list[tuple[str, Path, Path]]:
    cases: list[tuple[str, Path, Path]] = []
    for index, sentence in enumerate(app.CURRICULUM, start=1):
        stem = f"{index:02d}_{app.safe_tts_label(sentence)}"
        cases.append((sentence, COMMA_AUDIO_DIR / f"{stem}.wav", COMMA_AUDIO_DIR / f"{stem}_labels.txt"))
    return cases


In [ ]:
label_rows = []
for sentence, wav_path, label_path in comma_label_cases():
    labels = parse_audacity_labels(sentence, wav_path, label_path)
    expected_words = app.sentence_tts_words(sentence)
    observed_words = [label.word for label in labels]
    assert wav_path.exists(), wav_path
    assert label_path.exists(), label_path
    assert observed_words == expected_words, (sentence, observed_words, expected_words)

    for label in labels:
        label_rows.append(
            {
                "Sentence": label.sentence,
                "Word": label.word,
                "Start (s)": label.start,
                "End (s)": label.end,
                "Duration (s)": label.end - label.start,
                "WAV": str(label.wav_path.relative_to(REPO_ROOT)),
                "Labels": str(label.label_path.relative_to(REPO_ROOT)),
            }
        )

labels_df = pd.DataFrame(label_rows)
labels_df

In [ ]:
gap_rows = []
for sentence, wav_path, label_path in comma_label_cases():
    labels = parse_audacity_labels(sentence, wav_path, label_path)
    for boundary_index, (previous_label, next_label) in enumerate(zip(labels, labels[1:]), start=1):
        gap_start = previous_label.end
        gap_end = next_label.start
        gap_rows.append(
            {
                "Sentence": sentence,
                "Boundary": boundary_index,
                "Previous word": previous_label.word,
                "Next word": next_label.word,
                "Manual gap start (s)": gap_start,
                "Manual gap end (s)": gap_end,
                "Manual gap midpoint (s)": (gap_start + gap_end) / 2,
                "Manual gap duration (ms)": (gap_end - gap_start) * 1000,
            }
        )

gaps_df = pd.DataFrame(gap_rows)
gaps_df

## Scoring Method

For each adjacent pair of labeled words, the benchmark computes one predicted boundary. A hit means the predicted boundary lands anywhere inside the manual silence gap. The notebook also tracks mean absolute error to the gap midpoint, because two miss-heavy methods can still differ in how close they are to the labeled gap.

In [ ]:
@dataclass(frozen=True)
class BoundaryScore:
    hits: int
    total: int
    mean_abs_error_to_gap_midpoint: float
    max_error_seconds: float

    @property
    def accuracy(self) -> float:
        return self.hits / self.total if self.total else 1.0


def score_boundary_predictions(
    labels: list[WordLabel], predicted_timestamps: dict[str, tuple[float, float]]
) -> tuple[BoundaryScore, list[dict[str, object]]]:
    rows: list[dict[str, object]] = []
    hits = 0
    midpoint_errors: list[float] = []
    gap_errors: list[float] = []

    for boundary_index, (previous_label, next_label) in enumerate(zip(labels, labels[1:]), start=1):
        previous_timestamp = predicted_timestamps[previous_label.word]
        next_timestamp = predicted_timestamps[next_label.word]
        predicted_boundary = (previous_timestamp[1] + next_timestamp[0]) / 2.0
        gap_start = previous_label.end
        gap_end = next_label.start
        gap_midpoint = (gap_start + gap_end) / 2.0
        hit = gap_start <= predicted_boundary <= gap_end
        midpoint_error = abs(predicted_boundary - gap_midpoint)
        gap_error = 0.0 if hit else min(abs(predicted_boundary - gap_start), abs(predicted_boundary - gap_end))

        hits += int(hit)
        midpoint_errors.append(midpoint_error)
        gap_errors.append(gap_error)
        rows.append(
            {
                "Sentence": previous_label.sentence,
                "Boundary": boundary_index,
                "Previous word": previous_label.word,
                "Next word": next_label.word,
                "Predicted boundary (s)": predicted_boundary,
                "Manual gap start (s)": gap_start,
                "Manual gap end (s)": gap_end,
                "Hit manual gap": hit,
                "Abs error to midpoint (s)": midpoint_error,
                "Error to nearest gap edge (s)": gap_error,
            }
        )

    return (
        BoundaryScore(
            hits=hits,
            total=max(len(labels) - 1, 0),
            mean_abs_error_to_gap_midpoint=sum(midpoint_errors) / len(midpoint_errors) if midpoint_errors else 0.0,
            max_error_seconds=max(gap_errors) if gap_errors else 0.0,
        ),
        rows,
    )


## Competing Methods

`Current app alignment` is the repository's current `align_sentence_audio_words` path, which asks faster-whisper for word timestamps and then validates sequential target-word matches.

`Previous proportional splitter` reproduces the earlier fallback method, which divides the full audio duration by normalized word character counts. It does not listen to the waveform.

In [ ]:
def run_current_alignment(sentence: str, audio_bytes: bytes) -> dict[str, tuple[float, float]]:
    return app.align_sentence_audio_words(sentence, audio_bytes)


def run_previous_proportional(sentence: str, audio_bytes: bytes) -> dict[str, tuple[float, float]]:
    return app.proportional_word_timestamps(sentence, audio_bytes)


method_functions = {
    "Current app alignment": run_current_alignment,
    "Previous proportional splitter": run_previous_proportional,
}

comparison_rows = []
score_rows = []
for method_name, method_function in method_functions.items():
    for sentence, wav_path, label_path in comma_label_cases():
        audio_bytes = wav_path.read_bytes()
        labels = parse_audacity_labels(sentence, wav_path, label_path)
        timestamps = method_function(sentence, audio_bytes)
        score, rows = score_boundary_predictions(labels, timestamps)
        score_rows.append(
            {
                "Method": method_name,
                "Sentence": sentence,
                "Hits": score.hits,
                "Total": score.total,
                "Hit rate": score.accuracy,
                "Mean abs error to midpoint (s)": score.mean_abs_error_to_gap_midpoint,
                "Max error to gap edge (s)": score.max_error_seconds,
            }
        )
        for row in rows:
            comparison_rows.append({"Method": method_name, **row})

comparison = pd.DataFrame(comparison_rows)
comparison

## Aggregate Results

In [ ]:
summary = (
    comparison.groupby("Method", as_index=False)
    .agg(
        Hits=("Hit manual gap", "sum"),
        Total=("Hit manual gap", "count"),
        Mean_abs_error_s=("Abs error to midpoint (s)", "mean"),
        Max_gap_edge_error_s=("Error to nearest gap edge (s)", "max"),
    )
    .assign(Hit_rate=lambda df: df["Hits"] / df["Total"])
    [["Method", "Hits", "Total", "Hit_rate", "Mean_abs_error_s", "Max_gap_edge_error_s"]]
    .sort_values("Hit_rate", ascending=False)
)

summary.style.format(
    {
        "Hit_rate": "{:.1%}",
        "Mean_abs_error_s": "{:.4f}",
        "Max_gap_edge_error_s": "{:.4f}",
    }
).hide(axis="index")

Expected local result from the current repository on the committed manual labels:

| Method | Manual-gap hits | Hit rate | Mean boundary error | Max gap-edge error |
|---|---:|---:|---:|---:|
| Current app alignment | 1/13 | 7.7% | 0.1221s | 0.2405s |
| Previous proportional splitter | 0/13 | 0.0% | 0.2164s | 0.3640s |

## Interpretation

The current faster-whisper timestamp method is better than the previous proportional splitter, but the manual-label benchmark shows that it is still not good enough for precise clickable word clips. It lands only one of thirteen internal boundaries inside the labeled silence gaps.

That is an important data-science result, not just a unit-test result. It means the next candidate should be evaluated against this committed labeled dataset before it is wired into the app. A useful acceptance target would be near-perfect manual-gap hits on these four curriculum clips, then a larger labeled set once more generated sentence audio is available.

## Reproducibility Notes

The benchmark depends on local faster-whisper availability because it evaluates the current app alignment method. The proportional baseline and label parsing are deterministic. The same assertions are tracked in `test_word_boundary_benchmark.py`, while this notebook keeps the dataset table, per-boundary rows, and aggregate data-science view in one place.